In [ ]:
pip install librosa soundfile

In [ ]:
pip install resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.6 MB/s eta 0:00:00


In [ ]:
import os, kagglehub, librosa, pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

# 1. DOWNLOAD & MAP DATASET
print("🔍 Locating UrbanSound8K...")
path = kagglehub.dataset_download("chrisfilo/urbansound8k")

METADATA_PATH = ""
audio_map = {}

for root, dirs, files in os.walk(path):
    if "UrbanSound8K.csv" in files:
        METADATA_PATH = os.path.join(root, "UrbanSound8K.csv")
    for f in files:
        if f.endswith('.wav'):
            audio_map[f] = os.path.join(root, f)

metadata = pd.read_csv(METADATA_PATH)

# 2. STANDARDIZED EXTRACTION (The "Twins" Logic)
def extract_final_features(file_path):
    # Force 22050Hz - If the App and Colab match this, the AI stops being 'blind'
    audio, sr = librosa.load(file_path, sr=22050, duration=4.0)

    # Max Normalization: Makes quiet recordings as loud as the training data
    max_amp = np.max(np.abs(audio))
    if max_amp > 0:
        audio = audio / max_amp

    # Use explicit math parameters
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40, n_fft=2048, hop_length=512)
    return np.mean(mfccs.T, axis=0)

# 3. EXTRACTION LOOP
features, labels = [], []
print("🚀 Extracting features (This takes ~5-10 mins)...")

for index, row in tqdm(metadata.iterrows(), total=len(metadata)):
    real_path = audio_map.get(row['slice_file_name'])
    if real_path:
        try:
            data = extract_final_features(real_path)
            features.append(data)
            # Class 1 is car_horn in UrbanSound8K
            labels.append(1 if row['classID'] == 1 else 0)
        except:
            continue

X = np.array(features)
y = np.array(labels)

# 4. FAIR DATASET BALANCING (Oversampling)
X_safe, X_horn = X[y == 0], X[y == 1]
print(f"\n📊 RAW: {len(X_horn)} Horns vs {len(X_safe)} Safe sounds.")

print("⚖️ Balancing dataset for fairness...")
X_horn_up = resample(X_horn, replace=True, n_samples=len(X_safe), random_state=42)
X_bal = np.vstack((X_safe, X_horn_up))
y_bal = np.hstack((np.zeros(len(X_safe)), np.ones(len(X_safe))))

# 5. THE SCALER (Saves the 'Average' of the soundscape)
scaler = StandardScaler()
X_final = scaler.fit_transform(X_bal)

# Save Scaler Immediately
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved as scaler.pkl")

# 6. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X_final, y_bal, test_size=0.2, stratify=y_bal)

# 7. MODEL ARCHITECTURE
model = Sequential([
    Dense(256, input_shape=(40,), activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Use a slightly slower learning rate for better stability
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['accuracy'])

print("🧠 Training the 'Fair' Brain...")
model.fit(X_train, y_train, epochs=50, batch_size=64, validation_data=(X_test, y_test))

# 8. SAVE EVERYTHING
model.save('smart_helmet_weights.h5') # Saving as .h5 for easy local loading
print("\n🔥 DONE! Download 'smart_helmet_weights.h5' and 'scaler.pkl' from the sidebar.")

🔍 Locating UrbanSound8K...
Using Colab cache for faster access to the 'urbansound8k' dataset.
🚀 Extracting features (This takes ~5-10 mins)...


 41%|████      | 3554/8732 [02:28<02:51, 30.15it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
 95%|█████████▌| 8324/8732 [05:19<00:17, 23.57it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
 95%|█████████▌| 8328/8732 [05:20<00:14, 27.09it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1523
  warnings.warn(
100%|██████████| 8732/8732 [05:33<00:00, 26.18it/s]
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwa


📊 RAW: 429 Horns vs 8303 Safe sounds.
⚖️ Balancing dataset for fairness...
✅ Scaler saved as scaler.pkl
🧠 Training the 'Fair' Brain...
Epoch 1/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9067 - loss: 0.2318 - val_accuracy: 0.9862 - val_loss: 0.1204
Epoch 2/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9678 - loss: 0.0897 - val_accuracy: 0.9943 - val_loss: 0.0375
Epoch 3/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9832 - loss: 0.0524 - val_accuracy: 0.9958 - val_loss: 0.0207
Epoch 4/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9891 - loss: 0.0350 - val_accuracy: 0.9964 - val_loss: 0.0149
Epoch 5/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9916 - loss: 0.0266 - val_accuracy: 0.9973 - val_loss: 0.0130
Epoch 6/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9938 - loss: 0.0201 - val_accuracy: 0.9982 - val_loss: 0.0108
Epoch 7/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9958 - loss: 0.0149 - val_acc


🔥 DONE! Download 'smart_helmet_weights.h5' and 'scaler.pkl' from the sidebar.
